# Walmart PostgreSQL Data Warehouse Design

## Phase 1 — Data Engineering

### Milestone 1.5 — Database Design

## Business Objective

The Walmart M5 dataset is provided in a structure designed for a forecasting competition rather than for relational analytics.

The purpose of this notebook is to design a PostgreSQL data warehouse that supports:

- ETL pipelines
- SQL analytics
- Data quality validation
- Feature engineering
- Demand forecasting
- Pricing analysis
- API and dashboard queries

The warehouse will separate descriptive business entities into dimension tables and measurable events into fact tables.

## Why the Source Data Must Be Redesigned

The original sales files use a wide format in which each day is stored as a separate column:

`d_1`, `d_2`, `d_3`, ..., `d_1913`

This format is useful for the M5 competition but is inefficient for PostgreSQL because:

- New dates would require new columns.
- Time-series queries become difficult.
- Joining sales to prices, weather, and holidays is harder.
- Feature engineering with SQL window functions becomes less practical.
- The table would contain thousands of columns.

The sales data will therefore be transformed into a long format where each row represents the units sold for one product, at one store, on one date.

## Source-to-Warehouse Mapping

| Source | Warehouse Destination | Purpose |
|---|---|---|
| `calendar.csv` | `dim_calendar` | Dates, weekdays, events, SNAP indicators, and Walmart weeks |
| `sales_train_evaluation.csv` | `fact_sales` | Daily units sold by item, store, and date |
| `sell_prices.csv` | `fact_prices` | Weekly selling price by item and store |
| Sales identifiers | `dim_product` | Product, department, and category hierarchy |
| Sales identifiers | `dim_store` | Store and state hierarchy |
| Open-Meteo API | `fact_weather` | Daily weather by geographic area |
| FRED API | `fact_economic_indicator` | Economic observations such as CPI and unemployment |

## Proposed Warehouse Model

The warehouse will use a star-style design.

Dimension tables contain descriptive information:

- `dim_calendar`
- `dim_product`
- `dim_store`
- `dim_economic_series`

Fact tables contain measurable observations:

- `fact_sales`
- `fact_prices`
- `fact_weather`
- `fact_economic_indicator`

This structure reduces duplication and creates clear relationships for SQL analytics and machine learning.

## High-Level Relationship Diagram

```text
                    dim_product
                         │
                         │ item_id
                         ▼
dim_calendar ───────► fact_sales ◄─────── dim_store
     │                   │                   │
     │ date              │ item_id           │ store_id
     │                   │ store_id          │ state_id
     │                   │ date              │
     │                   ▼                   │
     └──────────────► fact_prices ◄──────────┘
                         │
                         │ wm_yr_wk
                         ▼
                    dim_calendar


dim_calendar ───────► fact_weather

dim_economic_series ─► fact_economic_indicator

## Table 1 — `dim_product`

This dimension contains one row for each unique Walmart item.

| Column | PostgreSQL Type | Constraint | Description |
|---|---|---|---|
| `item_id` | `VARCHAR(30)` | Primary Key | Unique anonymized product identifier |
| `dept_id` | `VARCHAR(20)` | Not Null | Department identifier |
| `cat_id` | `VARCHAR(20)` | Not Null | Category identifier |

The product information will be extracted from the identifier columns in the sales dataset.

## Table 2 — `dim_store`

This dimension contains one row for each store.

| Column | PostgreSQL Type | Constraint | Description |
|---|---|---|---|
| `store_id` | `VARCHAR(10)` | Primary Key | Unique anonymized store identifier |
| `state_id` | `VARCHAR(5)` | Not Null | State associated with the store |

The M5 dataset contains ten stores across California, Texas, and Wisconsin.

## Table 3 — `dim_calendar`

This dimension contains one row for each calendar date.

| Column | PostgreSQL Type | Constraint | Description |
|---|---|---|---|
| `date` | `DATE` | Primary Key | Actual calendar date |
| `d` | `VARCHAR(10)` | Unique, Not Null | M5 day identifier such as `d_1` |
| `wm_yr_wk` | `INTEGER` | Not Null | Walmart year-week identifier |
| `weekday` | `VARCHAR(10)` | Not Null | Weekday name |
| `wday` | `SMALLINT` | Not Null | Numeric weekday identifier |
| `month` | `SMALLINT` | Not Null | Calendar month |
| `year` | `SMALLINT` | Not Null | Calendar year |
| `event_name_1` | `VARCHAR(50)` | Nullable | Primary event name |
| `event_type_1` | `VARCHAR(30)` | Nullable | Primary event type |
| `event_name_2` | `VARCHAR(50)` | Nullable | Secondary event name |
| `event_type_2` | `VARCHAR(30)` | Nullable | Secondary event type |
| `snap_ca` | `BOOLEAN` | Not Null | California SNAP indicator |
| `snap_tx` | `BOOLEAN` | Not Null | Texas SNAP indicator |
| `snap_wi` | `BOOLEAN` | Not Null | Wisconsin SNAP indicator |

## Table 4 — `fact_sales`

This is the central fact table.

Each row represents the number of units sold for one item, at one store, on one date.

| Column | PostgreSQL Type | Constraint | Description |
|---|---|---|---|
| `date` | `DATE` | Foreign Key, Not Null | Sales date |
| `item_id` | `VARCHAR(30)` | Foreign Key, Not Null | Product identifier |
| `store_id` | `VARCHAR(10)` | Foreign Key, Not Null | Store identifier |
| `units_sold` | `INTEGER` | Not Null | Daily units sold |

### Proposed primary key

`(date, item_id, store_id)`

The original daily columns will be converted into rows during the transformation stage using a melt operation.

## Table 5 — `fact_prices`

This fact table stores weekly selling prices.

| Column | PostgreSQL Type | Constraint | Description |
|---|---|---|---|
| `store_id` | `VARCHAR(10)` | Foreign Key, Not Null | Store identifier |
| `item_id` | `VARCHAR(30)` | Foreign Key, Not Null | Product identifier |
| `wm_yr_wk` | `INTEGER` | Not Null | Walmart year-week identifier |
| `sell_price` | `NUMERIC(10,2)` | Not Null | Weekly selling price |

### Proposed primary key

`(store_id, item_id, wm_yr_wk)`

The Walmart year-week identifier allows prices to be connected to dates through `dim_calendar`.

## Table 6 — `fact_weather`

This fact table will store daily weather observations for each state or representative store location.

| Column | PostgreSQL Type | Constraint | Description |
|---|---|---|---|
| `date` | `DATE` | Foreign Key, Not Null | Observation date |
| `state_id` | `VARCHAR(5)` | Not Null | State identifier |
| `temperature_max` | `NUMERIC(6,2)` | Nullable | Maximum daily temperature |
| `temperature_min` | `NUMERIC(6,2)` | Nullable | Minimum daily temperature |
| `precipitation` | `NUMERIC(8,2)` | Nullable | Daily precipitation |
| `snowfall` | `NUMERIC(8,2)` | Nullable | Daily snowfall |
| `wind_speed_max` | `NUMERIC(6,2)` | Nullable | Maximum wind speed |

### Proposed primary key

`(date, state_id)`

## Table 7 — `dim_economic_series`

This dimension identifies economic measures obtained from FRED.

| Column | PostgreSQL Type | Constraint | Description |
|---|---|---|---|
| `series_id` | `VARCHAR(30)` | Primary Key | FRED series identifier |
| `series_name` | `VARCHAR(100)` | Not Null | Descriptive name |
| `frequency` | `VARCHAR(20)` | Not Null | Daily, weekly, monthly, or quarterly |
| `units` | `VARCHAR(50)` | Nullable | Measurement units |

## Table 8 — `fact_economic_indicator`

This fact table stores economic observations over time.

| Column | PostgreSQL Type | Constraint | Description |
|---|---|---|---|
| `series_id` | `VARCHAR(30)` | Foreign Key, Not Null | Economic-series identifier |
| `observation_date` | `DATE` | Not Null | Observation date |
| `value` | `NUMERIC(18,6)` | Nullable | Published indicator value |

### Proposed primary key

`(series_id, observation_date)`

## Primary and Foreign Keys

| Table | Primary Key |
|---|---|
| `dim_product` | `item_id` |
| `dim_store` | `store_id` |
| `dim_calendar` | `date` |
| `fact_sales` | `(date, item_id, store_id)` |
| `fact_prices` | `(store_id, item_id, wm_yr_wk)` |
| `fact_weather` | `(date, state_id)` |
| `dim_economic_series` | `series_id` |
| `fact_economic_indicator` | `(series_id, observation_date)` |

### Main foreign-key relationships

- `fact_sales.date` → `dim_calendar.date`
- `fact_sales.item_id` → `dim_product.item_id`
- `fact_sales.store_id` → `dim_store.store_id`
- `fact_prices.item_id` → `dim_product.item_id`
- `fact_prices.store_id` → `dim_store.store_id`
- `fact_weather.date` → `dim_calendar.date`
- `fact_economic_indicator.series_id` → `dim_economic_series.series_id`

## Planned Indexes

Indexes will improve frequent filtering and joining operations.

Planned indexes include:

- `fact_sales(item_id, date)`
- `fact_sales(store_id, date)`
- `fact_sales(date)`
- `fact_prices(item_id, store_id, wm_yr_wk)`
- `dim_calendar(wm_yr_wk)`
- `fact_weather(state_id, date)`
- `fact_economic_indicator(observation_date)`

Primary-key indexes will be created automatically by PostgreSQL.

## Data Grain

The grain defines what one row represents.

| Table | Grain |
|---|---|
| `dim_product` | One row per product |
| `dim_store` | One row per store |
| `dim_calendar` | One row per date |
| `fact_sales` | One row per date, item, and store |
| `fact_prices` | One row per Walmart week, item, and store |
| `fact_weather` | One row per date and state |
| `fact_economic_indicator` | One row per economic series and observation date |

Defining the grain prevents duplicate records and helps determine valid primary keys.

## Important Design Decision: Sales Table Size

The evaluation sales dataset contains approximately:

- 30,490 product-store series
- 1,941 daily sales columns

Transforming the wide dataset into long format may produce roughly 59 million sales records.

This volume is appropriate for a large-scale data engineering project, but development will initially use a smaller subset.

### Development Strategy

1. Test with one store.
2. Validate transformations and constraints.
3. Expand to multiple stores.
4. Load the complete dataset using chunked processing.
5. Measure runtime, memory consumption, and database size.

## Data Quality Rules

The warehouse will enforce the following rules:

- Product identifiers must exist in `dim_product`.
- Store identifiers must exist in `dim_store`.
- Sales dates must exist in `dim_calendar`.
- Daily unit sales cannot be negative.
- Selling prices must be greater than zero.
- Duplicate primary keys are not permitted.
- SNAP fields must contain valid Boolean values.
- API observation dates must be valid dates.
- Economic series must exist in `dim_economic_series`.

## Why This Design Supports Machine Learning

The normalized design allows the modeling dataset to be built by joining:

- Historical daily sales
- Weekly selling prices
- Calendar and holiday information
- SNAP indicators
- Weather observations
- Economic indicators
- Product hierarchy
- Store geography

SQL window functions can then create:

- Lagged demand
- Rolling averages
- Price changes
- Store-level averages
- Category-level averages
- Seasonal indicators

This warehouse therefore becomes the foundation for feature engineering, demand forecasting, and pricing simulation.

## Next Steps

The next milestone is to convert this design into executable PostgreSQL schema files.

The implementation will include:

- `CREATE TABLE` statements
- Primary-key and foreign-key constraints
- Indexes
- Database initialization scripts
- Validation queries

After the schema is created, the ETL pipeline will extract the Walmart M5 source files, transform the sales data from wide to long format, validate the records, and load them into PostgreSQL.